Dùng SMOTE với Random OverSampler để cân bằng dữ liệu

Load clean data

In [1]:
import pandas as pd 
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline 
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# to visualise al the columns in the dataframe
pd.pandas.set_option('display.max_columns', None)

In [2]:
application_train = pd.read_csv('/content/cleandata.csv')

Modeling

In [3]:
#Chia dữ liệu 
Fraud = application_train[application_train['TARGET']==1]
Valid = application_train[application_train['TARGET']==0]
outlier_fraction = len(Fraud)/float(len(Valid))

print('outlier_fraction for the whole dataset:')
print(outlier_fraction)
print("Fraud Cases : {}".format(len(Fraud)))
print("Valid Cases : {}".format(len(Valid)))

outlier_fraction for the whole dataset:
0.08781828601345662
Fraud Cases : 24825
Valid Cases : 282686


In [4]:
from sklearn.model_selection import train_test_split

#Create independent and Dependent Features
columns = application_train.columns.tolist()
# Filter the columns to remove data we do not want 
columns = [c for c in columns if c not in ["TARGET"]]
# Store the variable we are predicting 
target = "TARGET"
# Define a random state 
state = np.random.RandomState(42)
X = application_train[columns]
y = application_train[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(246008, 210) (246008,)
(61503, 210) (61503,)


Feature Selection

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

In [6]:
sel = SelectFromModel(RandomForestClassifier(n_estimators = 100, random_state=state))
sel.fit(X_train, y_train)

SelectFromModel(estimator=RandomForestClassifier(random_state=RandomState(MT19937) at 0x7ACC44EA1B40))

In [8]:
sel.get_support()

array([False, False,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True, False, False, False, False,  True,
        True, False, False, False, False, False,  True,  True,  True,
        True,  True,  True,  True,  True, False,  True,  True,  True,
        True,  True,  True, False,  True,  True, False,  True,  True,
        True, False,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False,  True,
        True,  True, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False,  True,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
        True, False, False, False, False, False, False, False, False,
       False, False,

In [9]:
selected_feat= X_train.columns[(sel.get_support())].tolist()
len(selected_feat)

42

In [10]:
print(selected_feat)

['FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'APARTMENTS_AVG', 'BASEMENTAREA_AVG', 'YEARS_BEGINEXPLUATATION_AVG', 'YEARS_BUILD_AVG', 'COMMONAREA_AVG', 'ENTRANCES_AVG', 'FLOORSMAX_AVG', 'FLOORSMIN_AVG', 'LANDAREA_AVG', 'LIVINGAPARTMENTS_AVG', 'LIVINGAREA_AVG', 'NONLIVINGAREA_AVG', 'TOTALAREA_MODE', 'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'DAYS_LAST_PHONE_CHANGE', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT', 'AMT_REQ_CREDIT_BUREAU_YEAR', 'NAME_FAMILY_STATUS_Married', 'OCCUPATION_TYPE_Laborers', 'CREDIT_INCOME_PERCENT', 'ANNUITY_INCOME_PERCENT', 'CREDIT_TERM', 'DAYS_EMPLOYED_PERCENT']


In [11]:

app_X_train = X_train.copy() #Lưu một bản sao trước khi drop 
app_X_test = X_test.copy()

In [12]:
X_train = X_train[selected_feat]
X_test = X_test[selected_feat]

In [13]:
print (X_train.shape, X_test.shape)

(246008, 42) (61503, 42)


Logistic Regression

In [14]:
from sklearn.linear_model import LogisticRegression
logistic_regressor = LogisticRegression(C = 2)

In [15]:
logistic_regressor.fit(X_train,y_train)

LogisticRegression(C=2)

In [16]:
y_pred = logistic_regressor.predict(X_test)

In [17]:
from sklearn.metrics import classification_report,accuracy_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix

n_errors = (y_pred != y_test).sum()
# Run Classification Metrics
print("{}: {}".format("Logistic Regression errors",n_errors))
print("Accuracy Score :")
print(accuracy_score(y_test,y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test,y_pred))
print("ROC AUC score is: ",roc_auc_score(y_test,y_pred))

Logistic Regression errors: 4972
Accuracy Score :
0.9191584150366648
Confusion matrix :
[[56531     0]
 [ 4972     0]]
Classification Report :
              precision    recall  f1-score   support

         0.0       0.92      1.00      0.96     56531
         1.0       0.00      0.00      0.00      4972

    accuracy                           0.92     61503
   macro avg       0.46      0.50      0.48     61503
weighted avg       0.84      0.92      0.88     61503

ROC AUC score is:  0.5


Random Forest - Bagging ensemble of Decision trees

In [18]:
from sklearn.ensemble import RandomForestClassifier
random_forest = RandomForestClassifier(n_estimators = 100, random_state = state, verbose = 1, n_jobs = -1)

In [19]:
random_forest.fit(X_train,y_train)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:   57.9s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  2.1min finished


RandomForestClassifier(n_jobs=-1,
                       random_state=RandomState(MT19937) at 0x7ACC44EA1B40,
                       verbose=1)

In [21]:
y_pred = random_forest.predict(X_test) 

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    0.6s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    1.3s finished


In [22]:
n_errors = (y_pred != y_test).sum()
# Run Classification Metrics
print("{}: {}".format("Random Forest errors",n_errors))
print("Accuracy Score :")
print(accuracy_score(y_test,y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test,y_pred))
print("ROC AUC score is: ",roc_auc_score(y_test,y_pred))

Random Forest errors: 4967
Accuracy Score :
0.9192397118839731
Confusion matrix :
[[56512    19]
 [ 4948    24]]
Classification Report :
              precision    recall  f1-score   support

         0.0       0.92      1.00      0.96     56531
         1.0       0.56      0.00      0.01      4972

    accuracy                           0.92     61503
   macro avg       0.74      0.50      0.48     61503
weighted avg       0.89      0.92      0.88     61503

ROC AUC score is:  0.502245466299021


XGBoosting Classifier

In [23]:
from xgboost import XGBClassifier

In [24]:
xgb_classifier = XGBClassifier(n_estimators=100,max_depth=5)

In [25]:
xgb_classifier.fit(X_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [26]:
y_pred = xgb_classifier.predict(X_test)

In [27]:
n_errors = (y_pred != y_test).sum()
# Run Classification Metrics
print("{}: {}".format("Extreme Gradient Boost errors",n_errors))
print("Accuracy Score :")
print(accuracy_score(y_test,y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test,y_pred))
print("ROC AUC score is: ",roc_auc_score(y_test,y_pred))

Extreme Gradient Boost errors: 4986
Accuracy Score :
0.9189307838642018
Confusion matrix :
[[56367   164]
 [ 4822   150]]
Classification Report :
              precision    recall  f1-score   support

         0.0       0.92      1.00      0.96     56531
         1.0       0.48      0.03      0.06      4972

    accuracy                           0.92     61503
   macro avg       0.70      0.51      0.51     61503
weighted avg       0.89      0.92      0.88     61503

ROC AUC score is:  0.5136339414823238


Balancing class - using RandomOverSampling

In [31]:
from imblearn.over_sampling import RandomOverSampler

In [32]:
os =  RandomOverSampler(sampling_strategy=1)


In [35]:
X_train_res, y_train_res = os.fit_resample(X_train, y_train)

In [36]:
X_train_res.shape,y_train_res.shape

((452310, 42), (452310,))

In [39]:
from collections import Counter
print('Original dataset shape {}'.format(Counter(y_train)))
print('Resampled dataset shape {}'.format(Counter(y_train_res)))

Original dataset shape Counter({0.0: 226155, 1.0: 19853})
Resampled dataset shape Counter({0.0: 226155, 1.0: 226155})


Logistic regression 

In [40]:
logistic_regressor.fit(X_train_res,y_train_res)

LogisticRegression(C=2)

In [41]:
y_pred = logistic_regressor.predict(X_test)

In [42]:
n_errors = (y_pred != y_test).sum()
# Run Classification Metrics
print("{}: {}".format("Logistic Regression errors",n_errors))
print("Accuracy Score :")
print(accuracy_score(y_test,y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test,y_pred))
print("ROC AUC score is: ",roc_auc_score(y_test,y_pred))

Logistic Regression errors: 22536
Accuracy Score :
0.6335788498122042
Confusion matrix :
[[36421 20110]
 [ 2426  2546]]
Classification Report :
              precision    recall  f1-score   support

         0.0       0.94      0.64      0.76     56531
         1.0       0.11      0.51      0.18      4972

    accuracy                           0.63     61503
   macro avg       0.52      0.58      0.47     61503
weighted avg       0.87      0.63      0.72     61503

ROC AUC score is:  0.5781667781991279


Random forest

In [43]:
random_forest.fit(X_train_res,y_train_res)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:  3.2min finished


RandomForestClassifier(n_jobs=-1,
                       random_state=RandomState(MT19937) at 0x7ACC44EA1B40,
                       verbose=1)

In [44]:
y_pred = random_forest.predict(X_test)

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    1.3s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    2.1s finished


In [45]:
n_errors = (y_pred != y_test).sum()
# Run Classification Metrics
print("{}: {}".format("Random Forest errors",n_errors))
print("Accuracy Score :")
print(accuracy_score(y_test,y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test,y_pred))
print("ROC AUC score is: ",roc_auc_score(y_test,y_pred))

Random Forest errors: 5020
Accuracy Score :
0.9183779653025056
Confusion matrix :
[[56346   185]
 [ 4835   137]]
Classification Report :
              precision    recall  f1-score   support

         0.0       0.92      1.00      0.96     56531
         1.0       0.43      0.03      0.05      4972

    accuracy                           0.92     61503
   macro avg       0.67      0.51      0.50     61503
weighted avg       0.88      0.92      0.88     61503

ROC AUC score is:  0.5121408816865558


XGBoost

In [46]:
X_train_res = pd.DataFrame(data=X_train_res, columns=selected_feat)

In [47]:
X_train_res.shape

(452310, 42)

In [49]:
xgb_classifier.fit(X_train_res,y_train_res)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [50]:
y_pred = xgb_classifier.predict(X_test)

In [51]:
n_errors = (y_pred != y_test).sum()
# Run Classification Metrics
print("{}: {}".format("Extreme Gradient Boost errors",n_errors))
print("Accuracy Score :")
print(accuracy_score(y_test,y_pred))
print("Confusion matrix :")
print(confusion_matrix(y_test, y_pred))
print("Classification Report :")
print(classification_report(y_test,y_pred))
print("ROC AUC score is: ",roc_auc_score(y_test,y_pred))

Extreme Gradient Boost errors: 17329
Accuracy Score :
0.7182413865990277
Confusion matrix :
[[40949 15582]
 [ 1747  3225]]
Classification Report :
              precision    recall  f1-score   support

         0.0       0.96      0.72      0.83     56531
         1.0       0.17      0.65      0.27      4972

    accuracy                           0.72     61503
   macro avg       0.57      0.69      0.55     61503
weighted avg       0.90      0.72      0.78     61503

ROC AUC score is:  0.6864979823044143


Balancing class - SMOTE + Tomek

In [28]:
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import NearMiss